# Can Spotify song titles predict mood on unseen titles?

Evaluation extension for [Kumru Alagur's final-year project](./FYP_KUMRUALAGUR.ipynb).

Compare the dataset's original mood labels with labels revised using the original project's title-keyword rules. Both evaluations use the same songs, train/test rows, text representation and model settings. The original labels are the dataset's annotations. The revised labels measure agreement with a target partly defined by the input text.

Place the original `spotify_songs_eda.csv` beside this notebook and run all cells. The CSV must contain `track_name` and `emotion`. Install pandas, scikit-learn and matplotlib in the notebook's Python environment.

In [ ]:
import re
import unicodedata
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC

dataset = Path("spotify_songs_eda.csv")
if not dataset.is_file():
    raise FileNotFoundError(f"Place the original CSV beside the notebook: {dataset.resolve()}")

songs = pd.read_csv(dataset, usecols=["track_name", "emotion"], dtype="string")
moods = ["happy", "love", "sad", "party"]
songs["original"] = songs["emotion"].str.strip().str.lower()
if not songs["original"].isin(moods).all():
    raise ValueError("The emotion column must contain only happy, love, sad and party.")

def normalise_title(title):
    title = unicodedata.normalize("NFKC", title).casefold()
    return " ".join(re.findall(r"[^\W_]+", title))

songs["title"] = songs["track_name"].fillna("").map(normalise_title)
print(f"Rows excluded because the title contains no words: {(songs['title'] == '').sum()}")
songs = songs.loc[songs["title"] != ""].reset_index(drop=True)

## Reproduce the project's relabelling rules

The keyword dictionary and relabelling function below come from the original notebook. The classifier uses the same normalised title text in both experiments, with no stop-word filtering.

In [ ]:
emotion_keywords_clean = {
    "happy": {"happy","joy","smile","sunshine","fun","bright","dancing","alive",
              "summer","glow","laughing","carefree","bliss","shining","golden",
              "positivity","joyful","cheerful","dream","daydream","lucky","grateful"},

    "love":  {"love","romance","soulmate","darling","kiss","forever","passion",
              "affection","chemistry","devotion","heartbeat","romantic","beloved",
              "sweetheart","adore","lover","fantasy"},

    "sad":   {"sad","lonely","broken","cry","tears","pain","aching","love","hurt","goodbye",
              "heartbreak","lost","alone","empty","numb","regrets","shattered",
              "forgotten","silence","grief","misery"},

    "party": {"party","dance","club","love","beat","drop","bass","tonight","wild",
              "crazy","banger","hype","groove","rave","festival","electric",
              "floor","vibing","hands","disco","remix","rave"}
}

MOOD_NAMES = {"love", "happy", "sad", "party"}


def score_and_label(title: str, original_emotion: str) -> str:
    tokens = re.sub(r"[^a-z\s]", "", title.lower()).split()
    token_set = set(tokens)

    # Pass 1 — exact mood name in title
    for token in tokens:
        if token in MOOD_NAMES:
            # Check if any other mood scores higher
            scores = {mood: len(token_set & kws) for mood, kws in emotion_keywords_clean.items()}
            scores[token] -= 1  # reduce the exact match mood by 1
            top_mood = max(scores, key=scores.get)
            if scores[top_mood] >= 2:
                return top_mood  # another mood wins
            return token  # exact match wins

    # Pass 2 — keyword scoring (minimum 2 matches)
    scores = {mood: len(token_set & kws) for mood, kws in emotion_keywords_clean.items()}
    max_score = max(scores.values())
    if max_score >= 2:
        return max(scores, key=scores.get)

    return original_emotion   
songs["revised"] = [
    score_and_label(title, mood)
    for title, mood in zip(songs["track_name"], songs["original"])
]
print(f"Labels changed: {(songs['original'] != songs['revised']).sum():,} / {len(songs):,}")
display(pd.crosstab(songs["original"], songs["revised"]))

## Compare both label sets on the same held-out titles

Use the first fold of a seeded five-fold split stratified on the original labels. Group rows by normalised title so repeats stay on one side of the split. Each pipeline learns its vocabulary from the training rows. Macro F1 gives all four moods equal weight. Confusion matrices show the share of each reference mood predicted as each category.

In [ ]:
titles = songs["title"]
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train, test = next(splitter.split(titles, songs["original"], groups=titles))
assert set(titles.iloc[train]).isdisjoint(titles.iloc[test])
for label in ["original", "revised"]:
    if any(set(songs[label].iloc[rows]) != set(moods) for rows in (train, test)):
        raise ValueError(f"The fixed split needs all four moods in both partitions for {label} labels.")

print(f"Training rows: {len(train):,} | Held-out rows: {len(test):,}")
models = {
    "Logistic regression": LogisticRegression(max_iter=1000, random_state=42),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(max_iter=2000, random_state=42),
    "Random forest": RandomForestClassifier(n_estimators=100, random_state=42),
}
results = {}
for name, model in models.items():
    results[name] = {}
    for label in ["original", "revised"]:
        pipeline = make_pipeline(
            TfidfVectorizer(ngram_range=(1, 2), max_features=5000, sublinear_tf=True,
                            token_pattern=r"(?u)\b\w+\b"),
            clone(model),
        )
        pipeline.fit(titles.iloc[train], songs[label].iloc[train])
        expected = songs[label].iloc[test]
        predicted = pipeline.predict(titles.iloc[test])
        results[name][label] = f1_score(expected, predicted, labels=moods, average="macro")
        print(f"{name} | {label.title()} labels")
        print(classification_report(expected, predicted, labels=moods, zero_division=0))
        ConfusionMatrixDisplay.from_predictions(
            expected, predicted, labels=moods, normalize="true", values_format=".2f",
            cmap="Blues", im_kw={"vmin": 0, "vmax": 1},
        ).ax_.set_title(f"{name} | {label.title()} labels")
        plt.show()

summary = pd.DataFrame.from_dict(results, orient="index").rename_axis("Model")
summary["revised − original"] = summary["revised"] - summary["original"]
display(summary.round(3))

A higher score on revised labels would indicate that those labels were easier to predict under this setup. It would not establish that they better reflect listeners' perceptions. The holdout tests unseen normalised titles. It can still contain the same artists or alternate versions with differently worded titles.